# PRCP-1025 – Flight Price Prediction

## Problem Statement
Flight ticket prices are difficult to predict because fares change according to airline,
route, travel date, departure time, arrival time, duration, number of stops and other factors.

### Project Tasks
1. Prepare a complete data analysis report on the given data.
2. Create a predictive model to help customers predict future flight prices and plan their journey.
3. Compare multiple models and identify a candidate model for production.
4. Report challenges faced, techniques used and reasons.

**Domain:** Travel / Aviation  
**Target variable:** `Price`  
**Machine Learning type:** Supervised Regression

## Important
Put the flight-price dataset in the same folder as this notebook or inside a `Data` folder.

The loader below automatically searches for CSV/XLSX files and checks for the expected `Price`
column, so you do not need to edit file paths manually.

In [ ]:
# Cell 3 – Imports

import os
import glob
import re
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

In [ ]:
# Cell 4 – Automatically find the dataset

preferred_names = [
    "Data_Train.xlsx",
    "Data_Train.csv",
    "flight_fare.xlsx",
    "flight_fare.csv",
    "flight-fare.xlsx",
    "flight-fare.csv",
    "train.xlsx",
    "train.csv"
]

search_folders = [
    ".",
    "./Data",
    "./data",
    "./Dataset",
    "./dataset"
]

candidates = []

for folder in search_folders:
    for filename in preferred_names:
        full_path = os.path.join(folder, filename)
        if os.path.isfile(full_path):
            candidates.append(full_path)

# If a standard filename was not found, search all CSV/XLSX files.
if not candidates:
    for pattern in ["./*.csv", "./*.xlsx", "./Data/*.csv", "./Data/*.xlsx",
                    "./data/*.csv", "./data/*.xlsx"]:
        candidates.extend(glob.glob(pattern))

# Remove duplicate paths while preserving order.
candidates = list(dict.fromkeys(candidates))

if not candidates:
    raise FileNotFoundError(
        "No CSV/XLSX dataset was found. Put the PRCP-1025 flight dataset "
        "beside this notebook or inside a Data folder."
    )

print("Candidate files found:")
for i, file in enumerate(candidates, 1):
    print(f"{i}. {file}")

In [ ]:
# Cell 5 – Load the file containing the Price column

def read_table(file_path):
    if file_path.lower().endswith(".xlsx"):
        return pd.read_excel(file_path)
    return pd.read_csv(file_path)

selected_path = None
df = None

for file in candidates:
    try:
        temp = read_table(file)
        normalized = {
            str(c).strip().lower().replace(" ", "_")
            for c in temp.columns
        }
        if "price" in normalized:
            selected_path = file
            df = temp.copy()
            break
    except Exception as e:
        print("Could not read:", file, "|", e)

if df is None:
    # Use the first readable file and show its columns.
    for file in candidates:
        try:
            df = read_table(file)
            selected_path = file
            break
        except Exception:
            pass

if df is None:
    raise ValueError("The available files could not be read as CSV/XLSX.")

print("Selected dataset:", selected_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
# Cell 6 – Clean column names

def clean_column_name(column):
    column = str(column).strip()
    column = re.sub(r"[^A-Za-z0-9]+", "_", column)
    column = re.sub(r"_+", "_", column)
    return column.strip("_")

df.columns = [clean_column_name(c) for c in df.columns]

# Standardize common target spelling.
price_candidates = [c for c in df.columns if c.lower() == "price"]

if not price_candidates:
    raise KeyError(
        "A 'Price' column was not found. Available columns are: "
        + str(df.columns.tolist())
    )

if price_candidates[0] != "Price":
    df.rename(columns={price_candidates[0]: "Price"}, inplace=True)

print("Cleaned columns:")
print(df.columns.tolist())

In [ ]:
# Cell 7 – Basic information

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
display(df.dtypes.to_frame("Data_Type"))

print("\nDataset information:")
df.info()

In [ ]:
# Cell 8 – Descriptive statistics

display(df.describe(include="all").T)

In [ ]:
# Cell 9 – Missing values

missing_report = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": df.isna().mean().mul(100)
}).sort_values("Missing_Count", ascending=False)

display(missing_report)

print("Total missing cells:", int(df.isna().sum().sum()))

In [ ]:
# Cell 10 – Duplicate records

duplicate_count = int(df.duplicated().sum())

print("Duplicate rows:", duplicate_count)

if duplicate_count > 0:
    display(df[df.duplicated(keep=False)].head(20))

In [ ]:
# Cell 11 – Target variable analysis

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

print("Missing Price values:", int(df["Price"].isna().sum()))
print("Minimum Price:", df["Price"].min())
print("Maximum Price:", df["Price"].max())
print("Mean Price:", df["Price"].mean())
print("Median Price:", df["Price"].median())

display(df["Price"].describe())

In [ ]:
# Cell 12 – Remove rows with missing target only

before = len(df)
df = df.dropna(subset=["Price"]).reset_index(drop=True)
after = len(df)

print("Rows removed because Price was missing:", before - after)
print("Final rows:", after)

In [ ]:
# Cell 13 – Price distribution

plt.figure(figsize=(10, 5))
sns.histplot(df["Price"], kde=True)
plt.title("Flight Price Distribution")
plt.xlabel("Price")
plt.ylabel("Number of Flights")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.boxplot(x=df["Price"])
plt.title("Flight Price Boxplot")
plt.xlabel("Price")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 – Price outlier report using IQR

q1 = df["Price"].quantile(0.25)
q3 = df["Price"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

price_outliers = df[
    (df["Price"] < lower_bound) |
    (df["Price"] > upper_bound)
]

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Potential Price outliers:", len(price_outliers))

In [ ]:
# Cell 15 – Identify categorical and numerical columns

categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

print("\nNumerical columns:")
print(numerical_columns)

In [ ]:
# Cell 16 – Categorical value counts

for column in categorical_columns:
    print("\n" + "=" * 70)
    print(column)
    print("Unique values:", df[column].nunique(dropna=True))
    display(df[column].value_counts(dropna=False).head(20))

In [ ]:
# Cell 17 – Airline analysis

if "Airline" in df.columns:
    airline_summary = (
        df.groupby("Airline")["Price"]
        .agg(["count", "mean", "median", "min", "max"])
        .sort_values("mean", ascending=False)
    )

    display(airline_summary)

    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x="Airline", y="Price")
    plt.xticks(rotation=45, ha="right")
    plt.title("Flight Price by Airline")
    plt.tight_layout()
    plt.show()
else:
    print("Airline column not available in this dataset.")

In [ ]:
# Cell 18 – Source and Destination analysis

for column in ["Source", "Destination"]:
    if column in df.columns:
        summary = (
            df.groupby(column)["Price"]
            .agg(["count", "mean", "median"])
            .sort_values("mean", ascending=False)
        )

        print("\n" + column)
        display(summary)

        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=column, y="Price")
        plt.xticks(rotation=45, ha="right")
        plt.title(f"Flight Price by {column}")
        plt.tight_layout()
        plt.show()
    else:
        print(column, "column not available.")

In [ ]:
# Cell 19 – Total Stops analysis

if "Total_Stops" in df.columns:
    print("Stop categories:")
    display(df["Total_Stops"].value_counts(dropna=False))

    stop_summary = (
        df.groupby("Total_Stops")["Price"]
        .agg(["count", "mean", "median"])
        .sort_values("mean", ascending=False)
    )

    display(stop_summary)

    plt.figure(figsize=(9, 5))
    sns.boxplot(data=df, x="Total_Stops", y="Price")
    plt.xticks(rotation=30)
    plt.title("Flight Price by Total Stops")
    plt.tight_layout()
    plt.show()
else:
    print("Total_Stops column not available.")

In [ ]:
# Cell 20 – Additional Information analysis

if "Additional_Info" in df.columns:
    info_summary = (
        df.groupby("Additional_Info")["Price"]
        .agg(["count", "mean", "median"])
        .sort_values("mean", ascending=False)
    )

    display(info_summary.head(20))
else:
    print("Additional_Info column not available.")

# Date, Time and Duration Feature Engineering

The raw flight dataset commonly stores date, time, duration and stop information as text.
These columns are transformed into numerical features before modeling.

In [ ]:
# Cell 22 – Create working copy and parse journey date

data = df.copy()

if "Date_of_Journey" in data.columns:
    data["Date_of_Journey"] = pd.to_datetime(
        data["Date_of_Journey"],
        dayfirst=True,
        errors="coerce"
    )

    data["Journey_Day"] = data["Date_of_Journey"].dt.day
    data["Journey_Month"] = data["Date_of_Journey"].dt.month
    data["Journey_Year"] = data["Date_of_Journey"].dt.year
    data["Journey_DayOfWeek"] = data["Date_of_Journey"].dt.dayofweek

display(data.head())

In [ ]:
# Cell 23 – Convert departure and arrival times to minutes

def time_to_minutes(series):
    text = series.astype(str).str.extract(r"(\d{1,2}:\d{2})")[0]
    parsed = pd.to_datetime(
        text,
        format="%H:%M",
        errors="coerce"
    )
    return parsed.dt.hour * 60 + parsed.dt.minute

if "Dep_Time" in data.columns:
    data["Departure_Minutes"] = time_to_minutes(
        data["Dep_Time"]
    )

if "Arrival_Time" in data.columns:
    data["Arrival_Minutes"] = time_to_minutes(
        data["Arrival_Time"]
    )

print("Time features created.")

In [ ]:
# Cell 24 – Convert Duration to minutes

def duration_to_minutes(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    hour_match = re.search(r"(\d+)\s*h", text)
    minute_match = re.search(r"(\d+)\s*m", text)

    hours = int(hour_match.group(1)) if hour_match else 0
    minutes = int(minute_match.group(1)) if minute_match else 0

    if hour_match or minute_match:
        return hours * 60 + minutes

    # Fallback for numeric duration values
    try:
        return float(text)
    except ValueError:
        return np.nan

if "Duration" in data.columns:
    data["Duration_Minutes"] = (
        data["Duration"].apply(duration_to_minutes)
    )

if "Duration_Minutes" in data.columns:
    display(
        data[["Duration", "Duration_Minutes"]].head(10)
    )

In [ ]:
# Cell 25 – Convert Total_Stops to numeric

if "Total_Stops" in data.columns:

    stop_text = (
        data["Total_Stops"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    stop_map = {
        "non-stop": 0,
        "non stop": 0,
        "nonstop": 0,
        "1 stop": 1,
        "2 stops": 2,
        "3 stops": 3,
        "4 stops": 4
    }

    data["Stops_Numeric"] = stop_text.map(stop_map)

    extracted = pd.to_numeric(
        stop_text.str.extract(r"(\d+)")[0],
        errors="coerce"
    )

    data["Stops_Numeric"] = (
        data["Stops_Numeric"].fillna(extracted)
    )

    display(
        data[["Total_Stops", "Stops_Numeric"]]
        .head(10)
    )

In [ ]:
# Cell 26 – Journey month and duration analysis

if "Journey_Month" in data.columns:

    monthly = (
        data.groupby("Journey_Month")["Price"]
        .mean()
    )

    display(monthly)

    plt.figure(figsize=(8, 5))
    monthly.plot(kind="bar")
    plt.title("Average Flight Price by Journey Month")
    plt.xlabel("Journey Month")
    plt.ylabel("Average Price")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

if "Duration_Minutes" in data.columns:

    plt.figure(figsize=(9, 5))
    sns.scatterplot(
        data=data,
        x="Duration_Minutes",
        y="Price",
        alpha=0.5
    )

    plt.title("Flight Duration vs Price")
    plt.xlabel("Duration in Minutes")
    plt.ylabel("Price")
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 27 – Numerical correlation analysis

numeric_data = data.select_dtypes(
    include=np.number
)

if "Price" in numeric_data.columns:

    correlations = (
        numeric_data.corr()["Price"]
        .sort_values(ascending=False)
        .to_frame("Correlation_with_Price")
    )

    display(correlations)

    plt.figure(figsize=(14, 9))
    sns.heatmap(
        numeric_data.corr(),
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0
    )
    plt.title("Numerical Feature Correlation Heatmap")
    plt.tight_layout()
    plt.show()

# Task 1 – Data Analysis Report

After executing the EDA cells, document observations from the actual outputs:
- Which airlines show different fare levels?
- Which source/destination locations have different average fares?
- How does number of stops relate to price?
- Are there journey-month differences?
- Does duration show a relationship with price?
- Are there missing values or duplicates?
- Which numerical variables have stronger correlation with Price?

The exact conclusions should be based on the generated tables and charts.

# Task 2 – Predictive Modeling

`Price` is a continuous target, so regression algorithms are used.

Models:
1. Linear Regression
2. Random Forest Regressor
3. Gradient Boosting Regressor
4. Extra Trees Regressor

Metrics:
- MAE – Mean Absolute Error
- RMSE – Root Mean Squared Error
- R² – coefficient of determination

For MAE and RMSE, lower values are better. For R², higher values are better.

In [ ]:
# Cell 30 – Prepare X and y

model_data = data.copy()

# Remove target and raw text representations after feature engineering.
drop_columns = [
    "Price",
    "Date_of_Journey",
    "Dep_Time",
    "Arrival_Time",
    "Duration",
    "Total_Stops"
]

# Route is high-cardinality and largely represented by Source/Destination.
if "Route" in model_data.columns:
    drop_columns.append("Route")

X = model_data.drop(
    columns=[c for c in drop_columns if c in model_data.columns],
    errors="ignore"
)

y = model_data["Price"].copy()

# Remove all-null columns.
all_null = [
    c for c in X.columns
    if X[c].isna().all()
]

if all_null:
    X = X.drop(columns=all_null)

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

In [ ]:
# Cell 31 – Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

In [ ]:
# Cell 32 – Preprocessing pipeline

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ],
    remainder="drop"
)

print("Preprocessing pipeline ready.")

In [ ]:
# Cell 33 – Define models

models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

print("Models defined:")
for name in models:
    print("-", name)

In [ ]:
# Cell 34 – Train models and evaluate

results = []
trained_models = {}
predictions = {}

for name, estimator in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator)
        ]
    )

    pipeline.fit(X_train, y_train)

    pred = pipeline.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )

    r2 = r2_score(
        y_test,
        pred
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    trained_models[name] = pipeline
    predictions[name] = pred

results_df = pd.DataFrame(results)

# Rank primarily by R², with RMSE as tie-breaker.
results_df = (
    results_df
    .sort_values(
        by=["R2", "RMSE"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

display(
    results_df.style.format({
        "MAE": "{:.2f}",
        "RMSE": "{:.2f}",
        "R2": "{:.4f}"
    })
)

# Model Comparison Report

The table above is the required model-comparison report.

- **MAE:** average absolute difference between actual and predicted fare.
- **RMSE:** gives more penalty to large prediction errors.
- **R²:** indicates how much variation in flight price is explained by the model.

The candidate model is selected from the observed holdout results. This is a model-development
comparison, not a guarantee of future real-world performance.

In [ ]:
# Cell 36 – Model comparison graphs

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=results_df, x="R2", y="Model", ax=ax)
ax.set_title("Model Comparison – R²")
ax.set_xlabel("R² - Higher is Better")
ax.set_ylabel("Model")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=results_df, x="RMSE", y="Model", ax=ax)
ax.set_title("Model Comparison – RMSE")
ax.set_xlabel("RMSE - Lower is Better")
ax.set_ylabel("Model")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=results_df, x="MAE", y="Model", ax=ax)
ax.set_title("Model Comparison – MAE")
ax.set_xlabel("MAE - Lower is Better")
ax.set_ylabel("Model")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 37 – Actual vs predicted plots

for name, pred in predictions.items():

    plt.figure(figsize=(7, 6))

    plt.scatter(
        y_test,
        pred,
        alpha=0.5
    )

    low = min(
        float(y_test.min()),
        float(np.min(pred))
    )

    high = max(
        float(y_test.max()),
        float(np.max(pred))
    )

    plt.plot(
        [low, high],
        [low, high],
        "--"
    )

    plt.xlabel("Actual Price")
    plt.ylabel("Predicted Price")
    plt.title(f"Actual vs Predicted – {name}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 38 – Select candidate production model

best_model_name = results_df.iloc[0]["Model"]
best_pipeline = trained_models[best_model_name]
best_prediction = predictions[best_model_name]

best_metrics = results_df.iloc[0]

print("Candidate Production Model:", best_model_name)
print(f"MAE  : {best_metrics['MAE']:.2f}")
print(f"RMSE : {best_metrics['RMSE']:.2f}")
print(f"R2   : {best_metrics['R2']:.4f}")

In [ ]:
# Cell 39 – Error analysis

error_analysis = pd.DataFrame({
    "Actual_Price": y_test.to_numpy(),
    "Predicted_Price": best_prediction
})

error_analysis["Absolute_Error"] = (
    error_analysis["Actual_Price"]
    - error_analysis["Predicted_Price"]
).abs()

error_analysis["Percentage_Error"] = np.where(
    error_analysis["Actual_Price"] != 0,
    error_analysis["Absolute_Error"]
    / error_analysis["Actual_Price"] * 100,
    np.nan
)

display(
    error_analysis
    .sort_values(
        "Absolute_Error",
        ascending=False
    )
    .head(20)
)

residuals = (
    error_analysis["Actual_Price"]
    - error_analysis["Predicted_Price"]
)

plt.figure(figsize=(9, 5))
sns.histplot(residuals, kde=True)
plt.axvline(0, linestyle="--")
plt.title("Residual Distribution – " + best_model_name)
plt.xlabel("Actual Price - Predicted Price")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 40 – Feature importance for tree models

if best_model_name in {
    "Random Forest",
    "Extra Trees",
    "Gradient Boosting"
}:

    fitted_preprocessor = (
        best_pipeline
        .named_steps["preprocessor"]
    )

    fitted_model = (
        best_pipeline
        .named_steps["model"]
    )

    feature_names = (
        fitted_preprocessor
        .get_feature_names_out()
    )

    feature_importance = pd.DataFrame({
        "Feature": feature_names,
        "Importance": fitted_model.feature_importances_
    }).sort_values(
        "Importance",
        ascending=False
    )

    top_features = feature_importance.head(20)

    display(top_features)

    plt.figure(figsize=(10, 7))
    sns.barplot(
        data=top_features,
        x="Importance",
        y="Feature"
    )
    plt.title(
        "Top 20 Features – " + best_model_name
    )
    plt.tight_layout()
    plt.show()

else:
    print(
        "The selected model is Linear Regression, "
        "so tree-based feature importance is not available."
    )

# Report on Challenges Faced

| Challenge | Technique Used | Reason |
|---|---|---|
| Date stored as text | Extracted day, month, year and day-of-week | Converts date information into useful numerical features |
| Departure/arrival time stored as text | Converted time into minutes | Makes time information suitable for ML |
| Duration stored as `h` and `m` text | Converted duration into total minutes | Creates a continuous numerical feature |
| Total stops stored as categories | Converted stop count into a numerical feature | Preserves the ordered nature of number of stops |
| Categorical variables | One-hot encoding | Converts categories into numerical model inputs |
| Missing values | Median/mode imputation | Preserves rows and avoids unnecessary data loss |
| Numerical feature scales | StandardScaler | Prevents large-scale variables from dominating scale-sensitive models |
| High-cardinality Route | Removed from baseline model | Source and Destination already capture major route information and reduce dimensionality |
| Different model behavior | Compared four regression algorithms | Provides objective model comparison |
| Large prediction errors | MAE, RMSE and residual analysis | Identifies average and large errors |

# Business Recommendations

1. Provide customers with an estimated fare rather than a guaranteed future ticket price.
2. Monitor airline, source, destination, stops, journey timing and duration.
3. Retrain the model periodically when new fare data becomes available.
4. Add booking lead time, seat availability, demand, seasonality and historical fare changes when such
   data is available.
5. Monitor MAE and RMSE after deployment.
6. Monitor changes in incoming data to identify data drift.

# Limitations

- Historical data cannot capture every future market event.
- Airline pricing may change due to demand, competition, promotions, fuel costs and operational events.
- A single train/test split gives only one estimate of generalization.
- Time-based validation would be useful when predicting genuinely future fares.
- The prediction should be treated as an estimate, not a guaranteed ticket price.

# Conclusion

This project completes the required workflow:

**Data Loading → Data Cleaning → Exploratory Data Analysis → Feature Engineering →
Preprocessing → Multiple Regression Models → Model Comparison → Error Analysis →
Candidate Model Selection → Challenges Report → Final Output**

The candidate model is selected using the measured test-set results generated by this notebook.

In [ ]:
# Cell 45 – Save all final outputs

os.makedirs("outputs", exist_ok=True)

results_df.to_csv(
    "outputs/flight_price_model_comparison.csv",
    index=False
)

joblib.dump(
    best_pipeline,
    "outputs/flight_price_prediction_model.pkl"
)

pd.DataFrame({
    "Feature": X.columns
}).to_csv(
    "outputs/flight_model_features.csv",
    index=False
)

final_summary = pd.DataFrame([{
    "Candidate_Model": best_model_name,
    "MAE": best_metrics["MAE"],
    "RMSE": best_metrics["RMSE"],
    "R2": best_metrics["R2"]
}])

final_summary.to_csv(
    "outputs/flight_price_final_summary.csv",
    index=False
)

print("Files saved successfully:")
print("1. outputs/flight_price_model_comparison.csv")
print("2. outputs/flight_price_prediction_model.pkl")
print("3. outputs/flight_model_features.csv")
print("4. outputs/flight_price_final_summary.csv")

In [ ]:
# Cell 46 – Final verification

print("=" * 70)
print("PRCP-1025 FLIGHT PRICE PREDICTION – COMPLETED")
print("=" * 70)
print("Dataset:", selected_path)
print("Dataset shape:", df.shape)
print("Candidate model:", best_model_name)
print("MAE:", round(float(best_metrics["MAE"]), 2))
print("RMSE:", round(float(best_metrics["RMSE"]), 2))
print("R2:", round(float(best_metrics["R2"]), 4))
print("=" * 70)